In [1]:
# import stuff
import os
import numpy as np
import time

from datasets import load_dataset
import dspy
import weaviate

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# load dataset
import pandas as pd

multihop_queries = pd.read_csv("irpapers-multihop-queries.csv")

multihop_queries.head(1)

,PDF ID 1,PDF ID 2,Question,Answer
0,7,11,Why might the fusion strategy proposed by Mack...,Mackie et al.'s fusion combines full GRF and P...


In [ ]:
multihop_queries = multihop_queries.to_dict(orient="records")

In [4]:
# define MultimodalRAG agent

In [4]:
import dspy

lm = dspy.LM(
    "openai/gpt-4.1"
)

dspy.configure(lm=lm)

lm("say hello")

['Hello! How can I help you today? 😊']

In [5]:
import dspy

dspy.__version__

'3.0.4'

In [6]:
# define RAG systems
import base64
from io import BytesIO
from PIL import Image
from typing import Any, Literal
from weaviate.classes.query import Filter

class GenerateAnswer(dspy.Signature):
    """Assess the context and answer the question."""

    question: str = dspy.InputField(description="The question to answer.")
    context: list[str] | list[dspy.Image] = dspy.InputField(description="The context to use to answer the question.")
    answer: str = dspy.OutputField(description="The answer to the question.")

class RAGSystem(dspy.Module):
    def __init__(self, collection: Any, images_or_text: Literal["images", "text"], k: int = 5):
        self.generate_answer = dspy.Predict(GenerateAnswer)
        self.collection = collection
        self.images_or_text = images_or_text
        self.k = k
    def _get_objects(self, question: str) -> list[str] | list[dspy.Image]:
        if self.images_or_text == "images":
            response = self.collection.query.near_text(
                query=question,
                return_properties=["base64_str"],
                limit=self.k
            )
            objects = []
            for o in response.objects:
                b64_str = o.properties["base64_str"]
                decoded_b64 = base64.b64decode(b64_str)
                pil_image = Image.open(BytesIO(decoded_b64))
                objects.append(dspy.Image(pil_image))
            return objects
        elif self.images_or_text == "text":
            response = self.collection.query.hybrid(
                query=question,
                return_properties=["content"],
                limit=self.k
            )
            objects = []
            for o in response.objects:
                objects.append(o.properties["content"])
            return objects
        
    def _fetch_oracle_context(
        self,
        oracle_context_id: str, 
    ) -> str | dspy.Image:
        if self.images_or_text == "images":
            response = self.collection.query.fetch_objects(
                filters=Filter.by_property("dataset_id").like(oracle_context_id),
                return_properties=["base64_str"]
            )
            b64_str = response.objects[0].properties["base64_str"]
            decoded_b64 = base64.b64decode(b64_str)
            pil_image = Image.open(BytesIO(decoded_b64))
            return dspy.Image(pil_image)
            
        elif self.images_or_text == "text":
            response = self.collection.query.fetch_objects(
                filters=Filter.by_property("dataset_id").like(oracle_context_id),
                return_properties=["content"]
            )
            return response.objects[0].properties["content"]

    def __call__(
        self, 
        question: str, 
        oracle_context_id: str = None
    ) -> str:
        if oracle_context_id is None:
            context = self._get_objects(question)
        else:
            context = self._fetch_oracle_context(oracle_context_id)
        return self.generate_answer(question=question, context=context)

In [7]:
weaviate_client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=weaviate.auth.AuthApiKey(os.environ["WEAVIATE_API_KEY"])
)

collection = weaviate_client.collections.get("IRPapersText_Default")

rag_system = RAGSystem(collection, "text")

rag_system(question="What is HyDE?")

Prediction(
    answer='HyDE stands for "Hypothetical Document Embeddings." It is a novel unsupervised method for dense retrieval in information retrieval systems. The core idea behind HyDE is to use a large instruction-following language model (LLM), such as InstructGPT, to generate a hypothetical document that answers a user\'s query.\n\nHere’s how HyDE works:\n\n1. **Generation Step:** For each input query, HyDE instructs a generative LLM to "write a document that answers the question." This produces a hypothetical document, which is not part of the actual corpus and may contain hallucinations or fictional details, but is intended to be representative of a relevant answer.\n\n2. **Encoding Step:** This hypothetical document is then passed through a contrastive encoder (like Contriever or mContriever, which are pre-trained without supervision) to generate a dense embedding (vector representation).\n\n3. **Retrieval Step:** HyDE then searches for real documents in the corpus whose emb

In [37]:
from openai import OpenAI
import json

client = OpenAI()

def run_agentic_search(query: str, rag_system) -> str:
    """
    Run a query through the function-calling agent that uses a RAG system.

    Args:
        query: The user's question to answer
        rag_system: A callable that takes a 'question' parameter and returns 
                   an object with an 'answer' attribute

    Returns:
        The final answer string from the model
    """
    tools = [
        {
            "type": "function",
            "function": {
                "name": "ask_search_agent",
                "description": "Ask the search agent a question to retrieve a researched answer.",
                "parameters": {
                    "type": "object",
                    "properties": {
                        "question": {
                            "type": "string",
                            "description": "The question to ask the search agent.",
                        },
                    },
                    "required": ["question"],
                },
            },
        },
    ]

    system_prompt = """You are an expert question-answering system specialized in Information Retrieval (IR) research.

You are connected to a database of Information Retrieval papers and must use this knowledge base to answer questions about IR concepts, techniques, and research.

Your task is to provide accurate, well-informed answers about topics such as:
- Retrieval methods (BM25, dense retrieval, neural IR, etc.)
- Ranking and re-ranking techniques
- Query expansion and reformulation
- Evaluation metrics (NDCG, MRR, MAP, etc.)
- Embeddings and representation learning for IR
- RAG (Retrieval-Augmented Generation) systems
- And other IR-related research topics

Guidelines:
- Always use the ask_search_agent tool to retrieve relevant information from the IR paper database
- Ground your answers in the retrieved research literature
- When applicable, cite specific techniques, methods, or findings from the papers
- Synthesize information from multiple sources into clear, comprehensive responses
- If the retrieved information is insufficient, acknowledge the limitations"""

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query}
    ]

    max_loops = 3
    for i in range(max_loops):
        response = client.chat.completions.create(
            model="gpt-5",
            messages=messages,
            tools=tools,
        )
        assistant_message = response.choices[0].message
        messages.append(assistant_message)

        # If tool calls exist, process and loop again
        if getattr(assistant_message, "tool_calls", None):
            for tool_call in assistant_message.tool_calls:
                if tool_call.function.name == "ask_search_agent":
                    function_args = json.loads(tool_call.function.arguments)
                    question = function_args.get("question")

                    # Execute the RAG system
                    rag_response = rag_system(question=question)

                    # Add the tool result to messages
                    messages.append({
                        "role": "tool",
                        "tool_call_id": tool_call.id,
                        "content": json.dumps({"answer": rag_response.answer})
                    })
            continue  # Loop – have model process the updated messages for a follow-up response
        # If no tool calls, return the agent's response
        return assistant_message.content
    # If max loops reached, return last assistant message (or a warning if desired)
    return assistant_message.content if 'assistant_message' in locals() else "No response could be generated in 5 iterations."

In [38]:
test_response = run_agentic_search("What is HyDE?", rag_system)

print(test_response)

HyDE (Hypothetical Document Embeddings) is a zero-shot dense retrieval method that replaces supervised training with an LLM-generated “hypothetical document.” Instead of embedding the query directly, HyDE prompts an instruction-following LLM (e.g., InstructGPT) to write a plausible answer-like document for the query; that synthetic document is then embedded with a contrastively trained document encoder (e.g., Contriever) and used to retrieve real corpus documents via vector similarity. The dense encoding acts as a lossy bottleneck that filters LLM hallucinations and aligns retrieval with corpus content.

Core method
- Generate: Prompt an instruction-following LLM to produce a hypothetical document that would answer or be relevant to the query.
- Encode: Embed the hypothetical document with a dense document encoder trained without labels (e.g., Contriever).
- Retrieve: Use the hypothetical document embedding to search the corpus (inner product or cosine similarity). Optionally generate 

In [25]:
# define LM as Judge alignment score
# llm as judge
class AssessAlignmentScore(dspy.Signature):
    """You are an expert grader assessing if a system's answer is semantically aligned with the correct answer.
    Only return True if the system answer has essentially the same meaning as the correct answer.
    If the system answer misses key aspects or meaning, return False.
    """

    question: str = dspy.InputField(description="The question asked.")
    system_answer: str = dspy.InputField(description="The answer generated by the system.")
    correct_answer: str = dspy.InputField(description="The reference answer containing the correct and complete information.")
    score: bool = dspy.OutputField(description="True if system_answer is equivalent in meaning to correct_answer, otherwise False.")

judge = dspy.Predict(AssessAlignmentScore)

test_question = "What is HyDE?"
correct_answer = "HyDE stands for Hypothetical Document Embeddings, a technique for improving retrieval in AI systems by generating hypothetical answers and using their embeddings."

# System answer missing key aspect (embeddings)
incorrect_answer = "HyDE is a technique for improving retrieval in AI systems by generating hypothetical answers."
# System answer rewords but covers all key ideas
acceptable_answer = "Hypothetical Document Embeddings (HyDE) is a method to help AI retrieval by creating hypothetical documents as sample answers and using their vector representations."

response = judge(question=test_question, system_answer=incorrect_answer, correct_answer=correct_answer)
print(response)
response = judge(question=test_question, system_answer=acceptable_answer, correct_answer=correct_answer)
print(response)

Prediction(
    score=False
)
Prediction(
    score=True
)


In [26]:
usage = response.get_lm_usage()

print(type(usage))

<class 'NoneType'>


In [40]:
# run experiment

alignment_scores = np.array([], dtype=np.float32)
output_tokens = np.array([], dtype=np.float32)

K = 3

rag_system = RAGSystem(collection, "text", k=5)

start = time.time()
for idx, query in enumerate(multihop_queries):
    test_query, ground_truth_answer = query["Question"], query["Answer"]
    '''
    qa_system_response = rag_system(
        question=test_query,
    )
    '''
    qa_system_response = run_agentic_search(test_query, rag_system)
    #usage_dict = qa_system_response.get_lm_usage()["openai/gpt-4.1"]
    #input_tokens = np.append(input_tokens, usage_dict["prompt_tokens"])
    #output_tokens = np.append(output_tokens, usage_dict["completion_tokens"])

    ensemble_votes = 0

    system_answer = qa_system_response.answer if isinstance(qa_system_response, dspy.Prediction) else qa_system_response
    for judge_predictions in range(K):
        lm_judge_response = judge(
            question=test_query,
            system_answer=system_answer,
            correct_answer=ground_truth_answer
        )
        if lm_judge_response.score:
            ensemble_votes += 1
    if ensemble_votes >= K / 2:
        alignment_scores = np.append(alignment_scores, 1)
    else:
        alignment_scores = np.append(alignment_scores, 0)

    print(qa_system_response)
    print("\n")
    print(f"LLM Judge Ratings: {ensemble_votes}/{K}")
    print("\n")

    if idx % 5 == 4:
        print(f"Processed {idx+1} queries in {time.time() - start} seconds...")
        print("Alignment score running mean:", alignment_scores.mean())
        #print("Input tokens running mean:", input_tokens.mean())
        #print("Output tokens running mean:", output_tokens.mean())
        
print(alignment_scores.mean())
#print(input_tokens.mean())
#print(output_tokens.mean())

/opt/homebrew/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content="[[ ## an...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(
/opt/homebrew/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='[[ ## sc...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedVal

Short answer: Because what helps a first-stage retriever’s recall does not necessarily help a second-stage cross-encoder’s precision.

Based on Li et al.’s findings:
- Query-level fusion/expansion is fragile for cross-encoders. Mackie et al.’s weighted RRF fusion of GRF+PRF is designed for first-stage retrieval lists. When applied by concatenating or otherwise injecting GRF/PRF expansions into the query for a cross-encoder, Li et al. report consistent drops in precision metrics (e.g., nDCG@10). Cross-encoders are sensitive to noisy, off-topic, or distribution-shifted tokens introduced by GRF/PRF.
- Recall gains don’t translate in the reranking regime. First-stage fusion mainly boosts recall@k. In second-stage reranking, the candidate set is fixed; the problem shifts to fine-grained precision. The same fusion does not address the precision-focused scoring of cross-encoders and can even distort their relevance judgments.
- Score/rank fusion needs to happen at the reranking output, not th

/opt/homebrew/lib/python3.11/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected 10 fields but got 6: Expected `Message` - serialized value may not be as expected [field_name='message', input_value=Message(content='[[ ## an...: None}, annotations=[]), input_type=Message])
  PydanticSerializationUnexpectedValue(Expected `StreamingChoices` - serialized value may not be as expected [field_name='choices', input_value=Choices(finish_reason='st...ider_specific_fields={}), input_type=Choices])
  return self.__pydantic_serializer__.to_python(


Short answer: The test-collection study found that LLM-generated query variants tend to be less diverse and less effective: they grow pools more slowly, retrieve fewer unique relevant documents, score lower on P@10/NDCG@10, and sometimes drift in intent—leading to more unjudged/off-topic results. BEQUE’s rejection sampling directly targets this by only keeping query rewrites that demonstrably improve retrieval (e.g., increase unique relevant coverage/recall or effectiveness) when run against a retrieval system. This filtering discards formulaic or off-topic LLM variants that don’t add new relevant documents, thereby mitigating the study’s key limitation (poor pool growth and effectiveness) and aligning generated variants with retrieval value rather than surface diversity.

Details grounded in the literature:
- “Can Generative LLMs Create Query Variants for Test Collections?” reports that LLM variants yield smaller, less diverse pools, lower P@10 and NDCG@10, higher internal similarity,

In [41]:
alignment_scores = np.array(alignment_scores)
#input_tokens = np.array(input_tokens)
#output_tokens = np.array(output_tokens)

print(alignment_scores.mean())
#print(input_tokens.mean())
#print(output_tokens.mean())

0.7
